In [ ]:
import librosa as lb
import numpy as np
import pickle
from eval_tools import getGroundTruthTimestamps
import utils.constants as constants
import plotly.graph_objs as go
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

### Load Alignments

In [ ]:
s = 25

dtw_hyp_file = f"experiments/DTW/s{s}/hyp.npy"
noa_hyp_file = f"experiments/NOA/s{s}/hyp.npy"
noa_tsm_file = f"experiments/NOA/s{s}/tsm.npy"

# load
dtw_hyp = np.load(dtw_hyp_file)
noa_hyp = np.load(noa_hyp_file)
noa_tsm = np.load(noa_tsm_file)

# load the ground truth timestamps
query_annot_file = f'scenarios/s{s}/query.beats'
ref_annot_file = f'scenarios/s{s}/ref.beats'
gt = getGroundTruthTimestamps(query_annot_file, ref_annot_file).T
gt = gt * 22050 / 512

In [ ]:
pair_txt_file = f'scenarios/s{s}/pair.txt'
pair_txt = open(pair_txt_file, 'r').readlines()
pair_txt = [line.strip().split() for line in pair_txt]
id1 = pair_txt[0][0]
id2 = pair_txt[0][1]

# load the chroma features
audio_path_1 = f'Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4/{id1}.wav'
audio_path_2 = f'Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4/{id2}.wav'

# load the features
f1 = np.load(f'features/chroma_stft_norm2/{id1}.npy')
f2 = np.load(f'features/chroma_stft_norm2/{id2}.npy')

### Plot Alignment Slices

In [ ]:
from noa import alignNOA
path_noa, D = alignNOA(f1, f2, weights=np.array([2,3,3]), return_D=True)
path_noa = path_noa * 22050 / 512

In [ ]:
from numba import njit
@njit
def normalize_D(D):
    """Normalize the cost matrix D by dividing each element by (i + j + 2)"""
    for i in range(D.shape[0]):
        for j in range(D.shape[1]):
            D[i, j] = D[i, j] / (i + j + 2)
normalize_D(D)

In [ ]:
from utils.analysis import plot_alignment_slice
plot_alignment_slice(D, path_noa, gt, frame_start=0, frame_end=3000)

In [ ]:
plot_alignment_slice(D2, dtw_hyp * 22050 / 512, gt, frame_start=100, frame_end=500, title="DTW Alignment (Query-Sliced) with Normalized Accumulated Cost")

### Analyze Spectral Flux for onsets

In [ ]:
from librosa.feature import chroma_stft
def chroma_stft_feature(audio_path):
    y, sr = lb.load(audio_path, sr=None)
    return chroma_stft(y=y, sr=sr, norm=2)

In [ ]:
f1_chroma = chroma_stft_feature(audio_path_1)

In [ ]:
# plot f1 on a heatmap, which has shape (12, 1000), and make the figure wider.
plt.figure(figsize=(24, 6))
plt.imshow(f1_chroma, aspect='auto')
plt.colorbar()
plt.show()

In [ ]:
# plot spectral flux for f1
spectral_flux = np.diff(f1_chroma, axis=1)

# set all the negative values to be 0
spectral_flux[spectral_flux < 0] = 0

plt.figure(figsize=(24, 6))
plt.imshow(spectral_flux[:, :400], aspect='auto')
plt.colorbar()
plt.show()

In [ ]:
# plot energy flux for f1, mark energy fluxes >1 with red color
energy_flux = np.sum(spectral_flux, axis=0)
plt.figure(figsize=(24, 6))
plt.plot(energy_flux, label='Energy Flux')
over_thresh = np.where(energy_flux > 0.5)[0]
plt.scatter(over_thresh, energy_flux[over_thresh], color='red', label='>1 Energy Flux')
plt.legend()
plt.show()

In [ ]:
# calcul the indices of energy fluxes >1
over_thresh_indices = np.where(energy_flux > 0.5)[0]

In [ ]:
over_thresh_times = over_thresh_indices * 512 / 22050

from pydub import AudioSegment
import numpy as np

def add_beeps(audio_path, times, beep_freq=1000, beep_duration_ms=100, output_path="output.wav"):
    """
    Adds beep sounds to an audio at specific times.

    Parameters:
        audio_path (str): Path to the input audio file.
        times (list of float): Times in seconds where beep should occur.
        beep_freq (int): Frequency of the beep in Hz.
        beep_duration_ms (int): Duration of the beep in milliseconds.
        output_path (str): Path to save the output audio.
    """
    # Load audio
    audio = AudioSegment.from_file(audio_path)

    # Generate beep sound
    sample_rate = 44100
    t = np.linspace(0, beep_duration_ms / 1000, int(sample_rate * beep_duration_ms / 1000), False)
    beep = np.sin(2 * np.pi * beep_freq * t) * 0.15  # amplitude 0.15 for less loudness
    beep = np.int16(beep * 32767)  # convert to 16-bit PCM
    beep_segment = AudioSegment(
        beep.tobytes(),
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )

    # Overlay beeps on original audio
    for time_sec in times:
        audio = audio.overlay(beep_segment, position=int(time_sec * 1000))

    # Export result
    audio.export(output_path, format="wav")

add_beeps(audio_path_1, over_thresh_times)

### Analyze Costs

In [ ]:
C = 1 - f1.T @ f2

In [ ]:
# plot C[0] on bar chart, horizontal being the index, vertical being the value
i = 2400
di = D[i]
# filter out inf
di = di[di != np.inf]
plt.figure(figsize=(10, 5))
plt.barh(range(len(di)), 1 - di)
plt.title(f'Costs for Frame {i}')
plt.xlabel('1 - Cost')
plt.ylabel('Index')
plt.show()

In [ ]:
import numpy as np
from scipy import stats

def estimate_gaussian_width(arr, threshold_ratio=0.01, min_points=5, baseline_percentile=10):
    """
    Estimate the standard deviation (width) of a Gaussian peak in an array.
    Handles Gaussians with baseline offset: y = y0 + A*exp(-(x-mu)²/(2σ²))
    
    Parameters:
    -----------
    arr : array-like
        Input array containing a Gaussian-like peak
    threshold_ratio : float, default=0.01
        Ratio of peak amplitude below which to stop expanding window (0.01 ≈ 3σ)
    min_points : int, default=5
        Minimum number of points required for fitting
    baseline_percentile : float, default=10
        Percentile to use for baseline estimation (uses values away from peak)
    
    Returns:
    --------
    sigma : float
        Estimated standard deviation of the Gaussian
        Returns np.nan if estimation fails
    mu : int
        Index of the peak
    r_squared : float
        R² value of the fit (quality metric)
    """
    
    arr = np.asarray(arr)
    
    # Step 1: Find the peak
    mu = np.argmax(arr)
    y_max = arr[mu]
    
    if not np.isfinite(y_max) or y_max <= 0:
        return np.nan, mu, 0.0, None, None, None
    
    # Estimate baseline from points far from peak (lower percentile of finite values)
    finite_mask = np.isfinite(arr)
    if np.sum(finite_mask) < min_points:
        return np.nan, mu, 0.0, None, None, None
    
    # Use points away from the peak region for baseline estimation
    dist_from_peak = np.abs(np.arange(len(arr)) - mu)
    far_from_peak = dist_from_peak > len(arr) * 0.1  # At least 10% away
    baseline_points = arr[finite_mask & far_from_peak]
    
    if len(baseline_points) > 0:
        y0 = np.percentile(baseline_points, baseline_percentile)
    else:
        # Fallback: use low percentile of all finite values
        y0 = np.percentile(arr[finite_mask], baseline_percentile)
    
    # Ensure baseline is less than peak
    y0 = min(y0, y_max * 0.5)
    
    # Peak amplitude above baseline
    amplitude = y_max - y0
    
    # Step 2: Define fitting window adaptively
    # Use a moving average or check consecutive points below threshold
    threshold = y0 + threshold_ratio * amplitude
    consecutive_below = 3  # Number of consecutive points below threshold to stop
    
    # Expand left from peak
    left_idx = mu
    below_count = 0
    while left_idx > 0 and below_count < consecutive_below:
        if not np.isfinite(arr[left_idx - 1]):
            break
        if arr[left_idx - 1] < threshold:
            below_count += 1
        else:
            below_count = 0
        left_idx -= 1
    left_idx += below_count  # Move back to last good position
    
    # Expand right from peak
    right_idx = mu
    below_count = 0
    while right_idx < len(arr) - 1 and below_count < consecutive_below:
        if not np.isfinite(arr[right_idx + 1]):
            break
        if arr[right_idx + 1] < threshold:
            below_count += 1
        else:
            below_count = 0
        right_idx += 1
    right_idx -= below_count  # Move back to last good position
    
    # Step 3: Collect valid points in window
    window = arr[left_idx:right_idx + 1]
    indices = np.arange(left_idx, right_idx + 1)
    
    # Filter valid points (finite and above threshold)
    valid_mask = np.isfinite(window) & (window > threshold)
    
    if np.sum(valid_mask) < min_points:
        return np.nan, mu, 0.0, left_idx, right_idx, y0
    
    y_valid = window[valid_mask]
    x_valid = indices[valid_mask]
    
    # Step 4: Log-linear regression
    # For y = y0 + A*exp(-(x-mu)²/(2σ²)), subtract baseline first
    # ln((y - y0) / A) = -(x - mu)² / (2σ²)
    y_adjusted = y_valid - y0
    
    # Ensure adjusted values are positive
    positive_mask = y_adjusted > 0
    if np.sum(positive_mask) < min_points:
        return np.nan, mu, 0.0, left_idx, right_idx, y0
    
    y_adjusted = y_adjusted[positive_mask]
    x_valid = x_valid[positive_mask]
    
    z = np.log(y_adjusted / amplitude)
    x_centered_sq = (x_valid - mu) ** 2
    
    # Linear regression: z = slope * x_centered_sq + intercept
    # slope = -1/(2σ²), so σ = sqrt(-1/(2*slope))
    slope, intercept, r_value, p_value, std_err = stats.linregress(x_centered_sq, z)
    
    # Step 5: Extract sigma and quality check
    if slope >= 0:  # Invalid: slope should be negative for Gaussian
        return np.nan, mu, 0.0, left_idx, right_idx, y0
    
    sigma = np.sqrt(-1.0 / (2.0 * slope))
    r_squared = r_value ** 2
    
    return sigma, mu, r_squared, left_idx, right_idx, y0

In [ ]:
def estimate_best_gaussian_width(D, i, threshold_ratio_range=(0.5, 0.95), num_ratios=10):
    best_sigma = np.nan
    best_mu = np.nan
    best_r_squared = 0.0
    best_threshold_ratio = 0.0
    best_left_idx = np.nan
    best_right_idx = np.nan
    best_y0 = np.nan
    one_minus_Di = 1-D[i]
    for threshold_ratio in np.linspace(threshold_ratio_range[0], threshold_ratio_range[1], num_ratios):
        sigma, mu, r_squared, left_idx, right_idx, y0 = estimate_gaussian_width(one_minus_Di, threshold_ratio=threshold_ratio)
        if r_squared > best_r_squared:
            best_sigma = sigma
            best_mu = mu
            best_r_squared = r_squared
            best_threshold_ratio = threshold_ratio
            best_left_idx = left_idx
            best_right_idx = right_idx
            best_y0 = y0
    return best_sigma, best_mu, best_r_squared, best_threshold_ratio, best_left_idx, best_right_idx, best_y0

In [ ]:
i = 10000
sigma, mu, r_squared, threshold_ratio, best_left_idx, best_right_idx, best_y0 = estimate_best_gaussian_width(D, i, num_ratios=100)
print(f"Estimated sigma: {sigma:.2f}, mu: {mu}, r_squared: {r_squared:.2f}, threshold_ratio: {threshold_ratio:.2f}, best_left_idx: {best_left_idx}, best_right_idx: {best_right_idx}, best_y0: {best_y0}")

In [ ]:
best_sigmas = []
for i in range(D.shape[0]):
    sigma, mu, r_squared, threshold_ratio, best_left_idx, best_right_idx, best_y0 = estimate_best_gaussian_width(D, i, num_ratios=10)
    best_sigmas.append(sigma)
plt.figure(figsize=(10, 5))
plt.plot(best_sigmas)
plt.show()

In [ ]:
# plot 1-C[2400][935:950]
plt.figure(figsize=(10, 5))
plt.plot(1-D[i])
# mark mu with a vertical line
plt.axvline(x=mu, color='red', linestyle='--', label=f'mu: {mu}')

# mark mu-sigma to mu+sigma with alpha=0.5 with a dashed line
if not np.isnan(mu) and not np.isnan(sigma):
    plt.axvline(x=mu - sigma, color='orange', linestyle='--', label=f'mu-sigma: {mu - sigma}')
    plt.axvline(x=mu + sigma, color='orange', linestyle='--', label=f'mu+sigma: {mu + sigma}')
    
# mark best_y0 with a horizontal line
plt.axhline(y=best_y0, color='purple', linestyle='--', label=f'best_y0: {best_y0}')
    
# mark best_left_idx to best_right_idx with a dashed line
if not np.isnan(best_left_idx) and not np.isnan(best_right_idx):
    plt.axvline(x=best_left_idx, color='green', linestyle='--', label=f'best_left_idx: {best_left_idx}')
    plt.axvline(x=best_right_idx, color='green', linestyle='--', label=f'best_right_idx: {best_right_idx}')

plt.legend(loc='lower right')
plt.show()